# Parte Comum: K-Means, Passada 2 (k fixo)

Dataset: Obesity (UCI 544), arquivo `data/parte_comum/obesity.csv`.

## Etapa 1: Padronização das variáveis

In [ ]:
# [IA] Claude Code – importacao das bibliotecas
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [ ]:
# [IA] Claude Code – leitura do dataset (caminho relativo a pasta do notebook)
CAMINHO_DADOS = Path("../../data/parte_comum/obesity.csv")

df = pd.read_csv(CAMINHO_DADOS)
print(df.shape)
df.head()

In [ ]:
# [IA] Claude Code – separar variaveis de entrada (X) e classe real (y)
# A classe real NAO entra no K-Means; fica guardada para o mapeamento e as metricas.
COLUNA_CLASSE = "NObeyesdad"

y = df[COLUNA_CLASSE].copy()
X = df.drop(columns=[COLUNA_CLASSE]).copy()

print("Classes reais:", y.nunique())
print(y.value_counts())

In [ ]:
# [IA] Claude Code – converter variaveis categoricas em numericas
# Binarias (yes/no e Gender) viram 0/1.
colunas_binarias = ["family_history_with_overweight", "FAVC", "SMOKE", "SCC"]
for coluna in colunas_binarias:
    X[coluna] = X[coluna].map({"no": 0, "yes": 1})

X["Gender"] = X["Gender"].map({"Female": 0, "Male": 1})

# Ordinais (frequencia): no < Sometimes < Frequently < Always.
ordem_frequencia = {"no": 0, "Sometimes": 1, "Frequently": 2, "Always": 3}
for coluna in ["CAEC", "CALC"]:
    X[coluna] = X[coluna].map(ordem_frequencia)

# Nominal sem ordem (meio de transporte): one-hot.
X = pd.get_dummies(X, columns=["MTRANS"], prefix="MTRANS", dtype=int)

# Conferencia: nenhuma categoria ficou sem conversao
assert X.isna().sum().sum() == 0, "Alguma categoria nao foi mapeada"
X.head()

In [ ]:
# [IA] Claude Code – padronizacao (z-score): media 0 e desvio padrao 1 em cada variavel
scaler = StandardScaler()
X_padronizado = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index,
)
X_padronizado.head()

In [ ]:
# [IA] Claude Code – conferir a padronizacao (media ~ 0 e desvio ~ 1)
conferencia = pd.DataFrame({
    "media_antes": X.mean(),
    "desvio_antes": X.std(ddof=0),
    "media_depois": X_padronizado.mean().round(6),
    "desvio_depois": X_padronizado.std(ddof=0).round(6),
})
conferencia

### Por que padronizar antes do K-Means?

_(texto do grupo)_